# Qwen3 Emotion Classifier — Kaggle free tier

- **Base model:** `Qwen/Qwen3-0.6B` (Apache-2.0, 128K context)
- **Data:** `dair-ai/emotion` — 16k/2k/2k, 6 classes, free
- **GPU:** T4x2 or P100 auto-assigned, 16GB VRAM, 9h session cap, 30 GPU-h/week
- **Method:** `AutoModelForSequenceClassification` + LoRA r=16

> **Run Cell 1 first, always.** Kaggle picks the GPU for you. The probe tells you
> whether you got a T4 or a P100, and the rest of the notebook branches on it.

> **Label order:** read from `features['label'].names` at runtime in Cell 4. The dataset
> card legend and the actual feature order disagree — hardcoding indices trains noise.

In [ ]:
# Cell 1 — hardware probe
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| n_gpu", torch.cuda.device_count())
# T4   = sm_75, compute 7.5, NO bfloat16 -> fp16 path
# P100 = sm_60, compute 6.0, NO bfloat16, NO 4-bit -> fp32 path
# USE_BF16 is derived below, never hardcoded True.

In [ ]:
# Cell 2 — installs
!pip install -q -U "transformers>=4.51" "peft>=0.14" "bitsandbytes>=0.45" \
               "datasets>=3.0" "accelerate>=1.0" scikit-learn evaluate
import transformers, peft, datasets
print("transformers", transformers.__version__, "| peft", peft.__version__)
# NOTE: eval_strategy= is the transformers v5 name. On v4.51 use evaluation_strategy=.

In [ ]:
# Cell 3 — config
import os, torch, random, numpy as np

MODEL_ID = "Qwen/Qwen3-0.6B"      # swap to Qwen3-1.7B if VRAM allows
DATASET  = "dair-ai/emotion"
MAX_LEN  = 160                      # audited in Cell 5, not guessed
EPOCHS   = 3
BS       = 8                       # per-device
ACCUM    = 2                       # effective batch 16
LR       = 2e-4
SEED     = 42
USE_BF16 = torch.cuda.is_bf16_supported()
OUT = "/kaggle/working/qwen3-emotion"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT, exist_ok=True)
print("bf16 enabled:", USE_BF16)

In [ ]:
# Cell 4 — data + label map read at runtime
from datasets import load_dataset
raw = load_dataset(DATASET)
print(raw)

# Never hardcode indices: the card legend and the feature order disagree.
id2label = {i: n for i, n in enumerate(raw["train"].features["label"].names)}
label2id = {n: i for i, n in id2label.items()}
print(id2label)

In [ ]:
# Cell 5 — length audit before spending GPU-hours
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:            # Qwen3 ships EOS-as-pad
    tok.pad_token = tok.eos_token
tok.padding_side = "right"           # required for last-token pooling

lens = [len(tok(t)["input_ids"]) for t in raw["train"]["text"][:4000]]
p = np.percentile(lens, [50, 90, 95, 99])
print(f"median {p[0]:.0f} | p90 {p[1]:.0f} | p95 {p[2]:.0f} | p99 {p[3]:.0f} | max {max(lens)}")
MAX_LEN = int(min(max(lens), 192))     # headroom over p99
cov = sum(l <= MAX_LEN for l in lens) / len(lens)
print(f"MAX_LEN -> {MAX_LEN} (covers {cov:.1%} of a 4k sample)")

In [ ]:
# Cell 6 — tokenize
def prep(batch):
    enc = tok(batch["text"], truncation=True, max_length=MAX_LEN)
    enc["labels"] = batch["label"]
    return enc

cols = raw["train"].column_names
tokd = raw.map(prep, batched=True, remove_columns=cols)
print(tokd)

In [ ]:
# Cell 7 — model + the pad_token fix
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding

load_kw = dict(attn_implementation="sdpa")

if USE_BF16:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=len(id2label), dtype=torch.bfloat16, **load_kw)
elif torch.cuda.get_device_capability(0)[0] >= 7:
    # T4 (sm_75): 4-bit NF4 works with bitsandbytes>=0.45
    from transformers import BitsAndBytesConfig
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=len(id2label), quantization_config=bnb, **load_kw)
else:
    # P100 (sm_60): 4-bit unsupported -> fp32 and a smaller batch
    print("!! sm_60/P100: 4-bit unsupported -> fp32, drop BS to 4")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=len(id2label), dtype=torch.float32, **load_kw)

# >>> THE LINE THAT PREVENTS SILENT GARBAGE <<<
# Qwen *ForSequenceClassification pools the LAST non-pad token. With pad_token_id
# unset it pools the last value in the row, which on a dynamically-padded batch is a
# <pad> position. Trains fine, plausible loss, garbage accuracy.
model.config.pad_token_id = tok.pad_token_id
assert model.config.pad_token_id is not None, "pooling would read pad as content"
model.config.id2label = id2label
model.config.label2id = label2id

for p_ in model.parameters():           # LoRA needs the base frozen
    p_.requires_grad = False

collator = DataCollatorWithPadding(tok, pad_to_multiple_of=8)
print("pad_token_id", model.config.pad_token_id, "| num_labels", model.config.num_labels)

In [ ]:
# Cell 8 — LoRA
from peft import LoraConfig, TaskType, get_peft_model

lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32,
                  lora_dropout=0.05, bias="none",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora)
trainable = sum(p_.numel() for p_ in model.parameters() if p_.requires_grad)
total     = sum(p_.numel() for p_ in model.parameters())
print(f"trainable {trainable:,} / {total:,} = {100*trainable/total:.2f}%")
model.print_trainable_parameters()

In [ ]:
# Cell 9 — metrics
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro"),
            "weighted_f1": f1_score(labels, preds, average="weighted")}

In [ ]:
# Cell 10 — train
from transformers import Trainer, TrainingArguments

args = dict(
    output_dir=f"{OUT}/ckpt",
    per_device_train_batch_size=BS,
    per_device_eval_batch_size=BS * 2,
    gradient_accumulation_steps=ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    group_by_length=True,
    seed=SEED,
    report_to=[],
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
)

trainer = Trainer(model=model, args=TrainingArguments(**args),
                  train_dataset=tokd["train"], eval_dataset=tokd["validation"],
                  data_collator=collator, compute_metrics=compute_metrics)
trainer.train()
print(trainer.evaluate(tokd["test"]))

In [ ]:
# Cell 11 — per-class report + confusion matrix
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

pred = trainer.predict(tokd["test"])
y, yh = pred.label_ids, pred.predictions.argmax(-1)
names = [id2label[i] for i in range(len(id2label))]
print(classification_report(y, yh, target_names=names, digits=3))

cm = confusion_matrix(y, yh)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(cm)), names, rotation=45, ha="right")
ax.set_yticks(range(len(cm)), names)
for i in range(len(cm)):
    for j in range(len(cm)):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im)
ax.set_title("Emotion confusion matrix (test)")
plt.tight_layout()
plt.savefig(f"{OUT}/cm.png", dpi=120)

In [ ]:
# Cell 12 — inference
clf = trainer.model.eval()

def predict(texts, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        b = tok(texts[i:i + bs], return_tensors="pt", padding=True,
                truncation=True, max_length=MAX_LEN).to(clf.device)
        with torch.no_grad():
            out += clf(**b).logits.argmax(-1).tolist()
    return out

tests = ["i can go from feeling so hopeless to so damned hopeful just from being around someone who cares",
         "this absolutely ruined my entire week",
         "i am so nervous about tomorrow i cannot sleep",
         "she planned a whole surprise party for me i am shocked"]
for t, p_ in zip(tests, predict(tests)):
    print(f"{id2label[p_]:>9} | {t[:60]}")

In [ ]:
# Cell 13 — save adapter + merged model
trainer.save_model(f"{OUT}/adapter")     # LoRA only: ~35MB for 0.6B
tok.save_pretrained(f"{OUT}/adapter")

merged = trainer.model.merge_and_unload()  # fuse LoRA into base weights
merged = merged.to(torch.float16)
merged.save_pretrained(f"{OUT}/merged", safe_serialization=True)  # ~1.2GB
tok.save_pretrained(f"{OUT}/merged")
print(os.listdir(OUT))

## Dataset swaps (all free)

- **GoEmotions** `google-research-datasets/go_emotions`, config `simplified` — 54.3k, Apache-2.0, **multi-label**. Needs BCEWithLogLoss + per-sample sigmoid instead of cross-entropy, or extract the first emotion for single-label.
- `mteb/tweet_eval`, config `emotion` — 6-class English tweets
- `cardiffnlp/tweet_xl`, config `en` — 11-class, includes neutral and irritation
- `IDEA-CCNL/EmoCacao` — Chinese, 6-class
- `dair-ai/emoji` — 64-class emoji sentiment

Do not mix a multi-label dataset into Cells 9–10 without swapping the loss.